# 04 — Modeling and Business Results

Run the full price elasticity pipeline, evaluate statistical quality, and translate estimates into business decisions.

**Reference:** Géron Ch.4 — *Training Models* (Linear Regression, OLS, bias/variance); Ch.2 — *Fine-Tune Your Model*, *Evaluate Your System on the Test Set*.

> *"In statistics, the term 'model' refers to any mathematical function that maps a set of input variables to an output variable."* — Géron Ch.4

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from price_elasticity.config import load_config
from price_elasticity.analysis import run_analysis

config = load_config(PROJECT_ROOT / 'configs' / 'project.toml')
config_path = PROJECT_ROOT / 'configs' / 'project.toml'

## 1. Run the Analysis Pipeline

Set `RUN_PIPELINE = True` to re-fit the model from scratch, or `False` to load cached results.

In [ ]:
RUN_PIPELINE = True  # Set to False to load cached results

if RUN_PIPELINE:
    result = run_analysis(config_path)
    print(f"Pipeline complete.")
    print(f"Products analyzed: {result['n_products']}")
    print(json.dumps(result.get('summary', {}), indent=2))
else:
    print('Using cached results. Set RUN_PIPELINE = True to re-run.')

## 2. Load Results

In [ ]:
results_path = PROJECT_ROOT / 'reports' / 'price_elasticity.csv'
df = pd.read_csv(results_path)
print(f'Products with elasticity estimate: {len(df):,}')
df.describe().round(3)

## 3. Elasticity Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

elast = df['price_elasticity'].clip(-5, 5)
axes[0].hist(elast, bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(-1, color='red', linestyle='--', linewidth=1.5, label='epsilon = -1 (unit elastic)')
axes[0].axvline(0, color='gray', linestyle='--', linewidth=1)
axes[0].set_title('Elasticity Distribution')
axes[0].set_xlabel('Price Elasticity')
axes[0].set_ylabel('Number of products')
axes[0].legend(fontsize=8)

axes[1].hist(df['r2'].clip(0, 1), bins=50, color='coral', edgecolor='white')
axes[1].axvline(0.10, color='red', linestyle='--', label='R2=0.10 (quality threshold)')
axes[1].set_title('R-squared Distribution')
axes[1].set_xlabel('R2')
axes[1].legend(fontsize=8)

hi_q = df[df['r2'] >= 0.10]
lo_q = df[df['r2'] < 0.10]
axes[2].scatter(lo_q['price_elasticity'].clip(-5, 5), lo_q['r2'], alpha=0.3, s=8,
                color='lightgray', label=f'R2<0.10 (n={len(lo_q)})')
axes[2].scatter(hi_q['price_elasticity'].clip(-5, 5), hi_q['r2'], alpha=0.5, s=8,
                color='steelblue', label=f'R2>=0.10 (n={len(hi_q)})')
axes[2].axvline(-1, color='red', linestyle='--', linewidth=1)
axes[2].set_title('Elasticity vs R2')
axes[2].set_xlabel('Price Elasticity')
axes[2].set_ylabel('R2')
axes[2].legend(fontsize=8)

plt.suptitle('Price Elasticity Model — Quality Assessment', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Elastic products (< -1):   {(df['price_elasticity'] < -1).sum():,}")
print(f"Inelastic products (>= -1): {(df['price_elasticity'] >= -1).sum():,}")
print(f"High R2 (>= 0.10):         {(df['r2'] >= 0.10).sum():,}")

## 4. Confidence Intervals — Model Uncertainty

A point estimate without uncertainty is incomplete. We show OLS analytical 95% CIs.

In [ ]:
ci_lo_col = [c for c in df.columns if 'ci_' in c and 'lower' in c]
ci_hi_col = [c for c in df.columns if 'ci_' in c and 'upper' in c]

if ci_lo_col and ci_hi_col:
    ci_lo, ci_hi = ci_lo_col[0], ci_hi_col[0]
    df['ci_width'] = df[ci_hi] - df[ci_lo]

    top = df[df['r2'] >= 0.10].nsmallest(20, 'ci_width').sort_values('price_elasticity')

    fig, ax = plt.subplots(figsize=(10, 7))
    y_pos = range(len(top))
    colors = ['steelblue' if e < -1 else 'coral' for e in top['price_elasticity']]
    ax.barh(
        list(y_pos), top['price_elasticity'],
        xerr=[top['price_elasticity'] - top[ci_lo], top[ci_hi] - top['price_elasticity']],
        height=0.6, color=colors, capsize=3, alpha=0.8
    )
    ax.axvline(-1, color='red', linestyle='--', linewidth=1.5, label='Unit elastic (e=-1)')
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels([p[:40] for p in top['product']], fontsize=8)
    ax.set_xlabel('Price Elasticity (95% CI)')
    ax.set_title('Top 20 Products by CI Precision (R2 >= 0.10)\nBlue = elastic, Coral = inelastic',
                 fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

print('CI width statistics:')
print(df['ci_width'].describe().round(3))

## 5. Revenue Simulation — H4: Non-linear Sweet Spot

In [ ]:
discount_range = np.linspace(-0.5, 0.3, 100)

scenarios = [
    {'label': 'Elastic (e = -2.0)', 'elasticity': -2.0, 'color': 'steelblue'},
    {'label': 'Unit-elastic (e = -1.0)', 'elasticity': -1.0, 'color': 'green'},
    {'label': 'Inelastic (e = -0.3)', 'elasticity': -0.3, 'color': 'coral'},
    {'label': 'Very inelastic (e = -0.1)', 'elasticity': -0.1, 'color': 'orange'},
]

fig, ax = plt.subplots(figsize=(11, 5))
for s in scenarios:
    e = s['elasticity']
    revenue_delta = [(1 + d) * (1 + e * d) - 1 for d in discount_range]
    ax.plot(discount_range * 100, [r * 100 for r in revenue_delta],
            label=s['label'], color=s['color'], linewidth=2)

ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Price Change (%)')
ax.set_ylabel('Revenue Change (%)')
ax.set_title('Revenue Impact vs Price Change by Elasticity Level', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Top Products — Decision Table

In [ ]:
quality = df[df['r2'] >= 0.10].copy()
elastic = quality[quality['price_elasticity'] < -1].sort_values('price_elasticity')
inelastic = quality[(quality['price_elasticity'] >= -1) & (quality['price_elasticity'] < 0)].sort_values('price_elasticity', ascending=False)

cols = ['product', 'price_elasticity', 'r2', 'p_value', 'observations', 'avg_price']
print('=== TOP 10 ELASTIC PRODUCTS (discount improves revenue) ===')
display(elastic[cols].head(10).round(3))

print('\n=== TOP 10 INELASTIC PRODUCTS (protect margin) ===')
display(inelastic[cols].head(10).round(3))

## 7. Portfolio Revenue Simulation at -10%

In [ ]:
scenario_discount = -0.10
portfolio = quality.copy()
portfolio['new_price'] = portfolio['avg_price'] * (1 + scenario_discount)
portfolio['new_demand'] = (portfolio['avg_demand'] * (1 + portfolio['price_elasticity'] * scenario_discount)).clip(lower=0)
portfolio['current_revenue'] = portfolio['avg_price'] * portfolio['avg_demand']
portfolio['new_revenue'] = portfolio['new_price'] * portfolio['new_demand']
portfolio['revenue_delta'] = portfolio['new_revenue'] - portfolio['current_revenue']

total_current = portfolio['current_revenue'].sum()
total_new = portfolio['new_revenue'].sum()

print(f'Scenario: {scenario_discount:.0%} price change across {len(portfolio)} products (R2 >= 0.10)')
print(f'Total current revenue (index): ${total_current:,.1f}')
print(f'Total new revenue (projected): ${total_new:,.1f}')
print(f'Portfolio revenue delta:       ${total_new - total_current:,.1f} ({(total_new/total_current - 1):.1%})')

## 8. Business Translation Summary

| Decision | Products | Action |
|----------|----------|--------|
| **Discount candidates** | e < -1, R2 >= 0.10, p < 0.05 | Run promotions — revenue grows |
| **Protect margin** | -1 < e < 0, R2 >= 0.10 | Avoid blanket discounts |
| **Insufficient data** | R2 < 0.10 or obs < 12 | Collect more price variation data |
| **Price increase test** | e near 0 | Run A/B test for small price increase |

**Next:** `05_deployment_and_consumption.ipynb` — Streamlit dashboard walkthrough and production pipeline.